## Inciso 4: Modelo de renovación urbana

### Variables de decisión

- $x_1$: número de unidades sencillas a construir
- $x_2$: número de unidades dobles a construir
- $x_3$: número de unidades triples a construir
- $x_4$: número de unidades cuádruples a construir
- $y$: número de casas populares a demoler ($0 \le y \le 300$)

### Función objetivo

Maximizar la recaudación de impuestos:

$$\text{Max } z = 1000x_1 + 1900x_2 + 2700x_3 + 3400x_4$$

### Restricciones

* Disponibilidad de terreno (el área demolida da $0.25y$ acres, de los cuales solo el 85% es utilizable para lotes, ya que 15% se destina a calles, áreas abiertas y servicios):

$$0.18x_1 + 0.28x_2 + 0.4x_3 + 0.5x_4 \le 0.85(0.25y)$$

* Límite de demolición:

$$y \le 300$$

* Financiamiento (costo de demolición + costo de construcción $\le \$15{,}000{,}000$):

$$2000y + 50000x_1 + 70000x_2 + 130000x_3 + 160000x_4 \le 15{,}000{,}000$$

* Mezcla de unidades

$$x_3 + x_4 \ge 0.25(x_1+x_2+x_3+x_4) \quad \text{(triples + cuádruples} \ge 25\%\text{)}$$

$$x_1 \ge 0.20(x_1+x_2+x_3+x_4) \quad \text{(sencillas} \ge 20\%\text{)}$$

$$x_2 \ge 0.10(x_1+x_2+x_3+x_4) \quad \text{(dobles} \ge 10\%\text{)}$$

* No negatividad:

$$x_1, x_2, x_3, x_4, y \ge 0$$

### Inciso b): Solución en variables continuas (usando JuMP)

In [1]:
using JuMP
using HiGHS

In [2]:
model_cont = Model(HiGHS.Optimizer)

@variable(model_cont, x1 >= 0)          # unidades sencillas
@variable(model_cont, x2 >= 0)          # unidades dobles
@variable(model_cont, x3 >= 0)          # unidades triples
@variable(model_cont, x4 >= 0)          # unidades cuádruples
@variable(model_cont, 0 <= y <= 300)    # casas demolidas

@objective(model_cont, Max, 1000x1 + 1900x2 + 2700x3 + 3400x4)

@constraint(model_cont, terreno,
    0.18x1 + 0.28x2 + 0.4x3 + 0.5x4 <= 0.85 * 0.25 * y)

@constraint(model_cont, financiamiento,
    2000y + 50000x1 + 70000x2 + 130000x3 + 160000x4 <= 15_000_000)

@constraint(model_cont, triples_cuadruples,
    x3 + x4 >= 0.25 * (x1 + x2 + x3 + x4))

@constraint(model_cont, sencillas_min,
    x1 >= 0.20 * (x1 + x2 + x3 + x4))

@constraint(model_cont, dobles_min,
    x2 >= 0.10 * (x1 + x2 + x3 + x4))

optimize!(model_cont)

println("Estado: ", termination_status(model_cont))

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: blastrampoline 
LP has 5 rows; 5 cols; 22 nonzeros
Coefficient ranges:
  Matrix  [1e-01, 2e+05]
  Cost    [1e+03, 3e+03]
  Bound   [3e+02, 3e+02]
  RHS     [2e+07, 2e+07]
Presolving model
5 rows, 5 cols, 22 nonzeros 0s
5 rows, 5 cols, 22 nonzeros 0s
Presolve reductions: rows 5(-0); columns 5(-0); nonzeros 22(-0) - Not reduced
Problem not reduced by presolve: solving the LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -7.8123054463e+01 Ph1: 5(12.599); Du: 4(78.1231) 0.0s
          4     3.4396515386e+05 Pr: 0(0) 0.0s

Model status        : Optimal
Simplex   iterations: 4
Objective value     :  3.4396515386e+05
P-D objective error :  8.4612606318e-17
HiGHS run time      :          0.01
Estado: OPTIMAL


In [3]:
println("Recaudación de impuestos óptima: \$", round(objective_value(model_cont), digits=2))
println()
println("Casas demolidas (y):        ", round(value(y), digits=2))
println("Unidades sencillas (x1):    ", round(value(x1), digits=2))
println("Unidades dobles (x2):       ", round(value(x2), digits=2))
println("Unidades triples (x3):      ", round(value(x3), digits=2))
println("Unidades cuádruples (x4):   ", round(value(x4), digits=2))
println()
total_unidades = value(x1)+value(x2)+value(x3)+value(x4)
println("Total de unidades:          ", round(total_unidades, digits=2))
println("Costo de construcción:      \$", round(50000*value(x1)+70000*value(x2)+130000*value(x3)+160000*value(x4), digits=2))
println("Costo de demolición:        \$", round(2000*value(y), digits=2))

Recaudación de impuestos óptima: $343965.15

Casas demolidas (y):        244.49
Unidades sencillas (x1):    35.83
Unidades dobles (x2):       98.53
Unidades triples (x3):      44.79
Unidades cuádruples (x4):   0.0

Total de unidades:          179.15
Costo de construcción:      $1.451102993e7
Costo de demolición:        $488970.07


### Inciso c): Solución en variables enteras

Se resuelve lo mismo, pero declarando las variables como enteras, porque en la realidad no tiene sentido construir o demoler una fracción de casa.

In [4]:
model_int = Model(HiGHS.Optimizer)

@variable(model_int, x1 >= 0, Int)
@variable(model_int, x2 >= 0, Int)
@variable(model_int, x3 >= 0, Int)
@variable(model_int, x4 >= 0, Int)
@variable(model_int, 0 <= y <= 300, Int)

@objective(model_int, Max, 1000x1 + 1900x2 + 2700x3 + 3400x4)

@constraint(model_int, terreno,
    0.18x1 + 0.28x2 + 0.4x3 + 0.5x4 <= 0.85 * 0.25 * y)

@constraint(model_int, financiamiento,
    2000y + 50000x1 + 70000x2 + 130000x3 + 160000x4 <= 15_000_000)

@constraint(model_int, triples_cuadruples,
    x3 + x4 >= 0.25 * (x1 + x2 + x3 + x4))

@constraint(model_int, sencillas_min,
    x1 >= 0.20 * (x1 + x2 + x3 + x4))

@constraint(model_int, dobles_min,
    x2 >= 0.10 * (x1 + x2 + x3 + x4))

optimize!(model_int)

println("Estado: ", termination_status(model_int))

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: blastrampoline 
MIP has 5 rows; 5 cols; 22 nonzeros; 5 integer variables (0 binary)
Coefficient ranges:
  Matrix  [1e-01, 2e+05]
  Cost    [1e+03, 3e+03]
  Bound   [3e+02, 3e+02]
  RHS     [2e+07, 2e+07]
Presolving model
5 rows, 5 cols, 22 nonzeros 0s
5 rows, 5 cols, 22 nonzeros 0s
Presolve reductions: rows 5(-0); columns 5(-0); nonzeros 22(-0) - Not reduced
Objective function is integral with scale 0.01

Solving MIP model with:
   5 rows
   5 cols (0 binary, 5 integer, 0 implied int., 0 continuous, 0 domain fixed)
   22 nonzeros
   Thread count 12 (of 24 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP

In [5]:
println("Recaudación de impuestos óptima (entera): \$", round(objective_value(model_int), digits=2))
println()
println("Casas demolidas (y):        ", value(y))
println("Unidades sencillas (x1):    ", value(x1))
println("Unidades dobles (x2):       ", value(x2))
println("Unidades triples (x3):      ", value(x3))
println("Unidades cuádruples (x4):   ", value(x4))

Recaudación de impuestos óptima (entera): $343700.0

Casas demolidas (y):        245.0
Unidades sencillas (x1):    36.0
Unidades dobles (x2):       98.0
Unidades triples (x3):      45.0
Unidades cuádruples (x4):   0.0


### Comparación de resultados

| | Continuo | Entero |
|---|---|---|
| $x_1$ | 35.83 | 36 |
| $x_2$ | 98.53 | 98 |
| $x_3$ | 44.79 | 45 |
| $x_4$ | 0.0 | 0 |
| $y$   | 244.49 | 245 |
| $z$ (impuestos) | \$343,965.15 | \$343,700.00 |


Como se ve, prácticamente ambos resultados llegaron a lo mismo, pues la versión entera/discreta es la igual a redondear a mano los valores encontrados por medio de la versión secuencial del optimizador. 

Las restricciones de mezcla de unidades (sencillas ≥ 20%, dobles ≥ 10%, triples+cuádruples ≥ 25%) y la restricción de terreno disponible resultan activas (binding) en la solución continua, lo que sugiere que estas son las que realmente limitan la recaudación de impuestos, y no el financiamiento, que es lo que a simple vista parecería ser lo más determinante en este problema. 

Asimismo, en términos de recaudación de impuestos, aunque en la teoría se recauda más (aprox $200) en la versión continua que la entera, en la realidad esto no sería posible, justo por lo mencionado en la celda anterior, que en la práctica, no se puede construir o demoler únicamente una fracción de una casa. 